In [15]:
import os
from huggingface_hub import login
from dotenv import dotenv_values
import pandas as pd
from transformers import AutoConfig, AutoModel
import torchaudio
import torch
import torch.nn as nn
import numpy as np
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler

%load_ext tensorboard

login(dotenv_values()['HUGGINGFACE_API_KEY'])

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [16]:
csv_path   = '../data/SEP-28k-Extended_clips.csv'
clips_path = '../data/SEP-28k_CLIP'
print("csv exist:", os.path.exists(csv_path))
print("Clips exist:", os.path.exists(clips_path))

csv exist: True
Clips exist: True


In [17]:
# Check CSV
print("CSV exists:", os.path.exists(csv_path))

# Check one audio clip
path = clips_path + '/HeStutters/10/HeStutters_10_0.wav'
print("Data Exists:", os.path.exists(path))


CSV exists: True
Data Exists: True


In [18]:
# Load CSV
df = pd.read_csv(csv_path)

# Build filepath — Show/EpId/Show_EpId_ClipId.wav
df['filepath'] = df.apply(
    lambda row: os.path.join(
        clips_path,
        str(row['Show']),
        str(row['EpId']),
        f"{row['Show']}_{row['EpId']}_{row['ClipId']}.wav"
    ), axis=1
)

# Check how many files exist
df['exists'] = df['filepath'].apply(os.path.exists)
print(f"Total clips in CSV:     {len(df)}")
print(f"Clips found on disk:    {df['exists'].sum()}")
print(f"Clips NOT found:        {(~df['exists']).sum()}")

# Keep only existing files
df = df[df['exists']]
print("\nSample filepath:")
print(df['filepath'].iloc[0])

Total clips in CSV:     28177
Clips found on disk:    28177
Clips NOT found:        0

Sample filepath:
../data/SEP-28k_CLIP\HeStutters\0\HeStutters_0_0.wav


In [19]:
stutter_cols = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']

def create_soft_labels(row):
    return {
        'prolongation': row['Prolongation'] / 3.0,
        'block':        row['Block']        / 3.0,
        'soundrep':     row['SoundRep']     / 3.0,
        'wordrep':      row['WordRep']      / 3.0,
        'interjection': row['Interjection'] / 3.0,
        'fluent':       row['NoStutteredWords'] / 3.0
    }

soft_labels_df = df.apply(create_soft_labels, axis=1, result_type='expand')
df = pd.concat([df, soft_labels_df], axis=1)

print(df.head())

print("\nAverage probability per stutter type:")
for col in ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']:
    print(f"{col}: {df[col].mean():.3f}")

print("\nSample soft labels:")
print(df[['Show', 'EpId', 'ClipId',
          'prolongation', 'block',
          'soundrep', 'wordrep',
          'interjection', 'fluent']].head())

         Show  EpId  ClipId     Start      Stop is_probably_host  \
0  HeStutters     0       0  31900320  31948320             True   
1  HeStutters     0       1  31977120  32025120             True   
2  HeStutters     0       2  34809760  34857760             True   
3  HeStutters     0       3  35200640  35248640             True   
4  HeStutters     0       4  35721920  35769920             True   

        speaker  clip_silhouette_score SEP12k SEP28k-E  ... Music NoSpeech  \
0  Pamela Mertz               0.565847    NaN    train  ...     0        0   
1  Pamela Mertz               0.440298    NaN    train  ...     0        0   
2  Pamela Mertz               0.523960    NaN    train  ...     0        0   
3  Pamela Mertz               0.377892    NaN    train  ...     0        0   
4  Pamela Mertz               0.553643    NaN    train  ...     0        0   

                                            filepath  exists  prolongation  \
0  ../data/SEP-28k_CLIP\HeStutters\0\HeStutt

In [20]:
subset_fraction = 1
train_df = df[df['SEP28k-E']== 'train'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)
valid_df = df[df['SEP28k-E']== 'dev'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)
test_df = df[df['SEP28k-E']== 'test'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [22]:
# load voc2vec Encoder
print("\n Loading voc2vec encoder...")
# MODEL_NAME = "alkiskoudounas/voc2vec-hubert-ls-pt" # facebook/wav2vec2-base, alkiskoudounas/voc2vec-hubert-ls-pt, openai/whisper-large-v3-turbo
MODEL_NAME = "facebook/wav2vec2-base"
# MODEL_NAME = "openai/whisper-large-v3-turbo"
encoder_config = AutoConfig.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME, device_map="cuda")


 Loading voc2vec encoder...


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 2176.58it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
def convo_audio_data(filename):
  try:
    waveform,sample_rate=torchaudio.load(filename)
    #ensures both input and encoder are on the same device
    waveform = waveform.to(device)
    # Stereo to mono
    if waveform.shape[0] > 1:
      waveform = torch.mean(waveform,dim=0,keepdim=True)
    # Resample
    if sample_rate != 16000:
      waveform = torchaudio.functional.sample(
          waveform,
          sample_rate,
          16000,
      )
    # extract features
    emission = encoder(waveform.to(device)).last_hidden_state
    emission = emission.detach().cpu().numpy()
    return emission
  except Exception as e:
    print(f"Error processing {filename}: {e}")

In [24]:
# Hyper Parameters
batch_size    = 256
num_epochs    = 150
learning_rate = 0.0001

def load_data(dataframe, desc):
    x           = []
    y           = []
    names       = []
    discarded   = 0

    for _, row in tqdm(dataframe.iterrows(), desc=desc, total=len(dataframe)):
        
        # Load raw audio
        waveform, sample_rate = torchaudio.load(row['filepath'])

        name = row['Show'] + "_" + str(row['EpId']) + "_" + str(row['ClipId'])

        if waveform.size()[1] == 48000: # only if the clip is exactly the right size
            # Stereo to mono
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)

            # Resample to 16000Hz
            if sample_rate != 16000:
                waveform = torchaudio.functional.resample(
                    waveform, sample_rate, 16000
                )

            # append to dataset
            x.append(waveform)
            names.append(name)

            # Soft labels
            y.append({
                'prolongation': torch.tensor(float(row['prolongation']), dtype=torch.float32),
                'block':        torch.tensor(float(row['block']),        dtype=torch.float32),
                'soundrep':     torch.tensor(float(row['soundrep']),     dtype=torch.float32),
                'wordrep':      torch.tensor(float(row['wordrep']),      dtype=torch.float32),
                'interjection': torch.tensor(float(row['interjection']), dtype=torch.float32),
            })

        else: # do not append anything in this case
            discarded += 1
            # print('Rejecting clip: ' + name + ': expected length 48000 but got ' + str(waveform.size()[1]))
        
    print(f"Discarded in {desc}: {discarded}")
    return x, y, names

In [25]:
# Dataset Class for Soft Labels
class AudioDataset(Dataset):
    def __init__(self, x, y, n_samples, names):
        self.x         = x
        self.y         = y
        self.n_samples = n_samples
        self.names = names

    def __getitem__(self, index):
        # Convert embedding to tensor
        return self.x[index], self.y[index], self.names[index]

    def __len__(self):
        return self.n_samples

print("Loading Data to Device...")
x_train, y_train, names_train = load_data(train_df, "Train")
x_valid, y_valid, names_valid = load_data(valid_df, "Dev")
x_test,  y_test, names_test  = load_data(test_df,  "Test")

# Create Datasets
n_samples_train = len(x_train)
n_samples_valid = len(x_valid)
n_samples_test  = len(x_test)

print(f'\nTrain samples: {n_samples_train}')
print(f'Dev samples:   {n_samples_valid}')
print(f'Test samples:  {n_samples_test}')

train_dataset = AudioDataset(x_train, y_train, n_samples_train, names=names_train)
valid_dataset = AudioDataset(x_valid, y_valid, n_samples_valid, names=names_valid)
test_dataset  = AudioDataset(x_test,  y_test,  n_samples_test, names=names_test)

#Create DataLoaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)
valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
)

print("\nData loaders")
print(f"Train batches: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")
print(f"Test batches:  {len(test_loader)}")

Loading Data to Device...


Train: 100%|██████████| 15213/15213 [00:48<00:00, 312.17it/s]


Discarded in Train: 7


Dev: 100%|██████████| 6402/6402 [00:20<00:00, 313.80it/s]


Discarded in Dev: 9


Test: 100%|██████████| 6562/6562 [00:21<00:00, 302.94it/s]

Discarded in Test: 6

Train samples: 15206
Dev samples:   6393
Test samples:  6556

Data loaders
Train batches: 238
Valid batches: 100
Test batches:  103


In [26]:
# Model definition

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super(AttentionPooling, self).__init__()
        # Attention layer
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, x):
        scores  = self.attention(x)                    # → (batch, time_steps, 1)
        weights = torch.softmax(scores, dim=1)         # → (batch, time_steps, 1)
        pooled  = torch.sum(x * weights, dim=1)        # → (batch, hidden_size)
        return pooled


class StutterNet(nn.Module):
    def __init__(self, encoder_type='wav2vec'):
        super(StutterNet, self).__init__()

        if encoder_type == 'voc2vec':
            self.encoder = AutoModel.from_pretrained("alkiskoudounas/voc2vec-hubert-ls-pt", device_map="cuda")
            for param in self.encoder.parameters():
                param.requires_grad = False
        
        if encoder_type == 'wav2vec':
            self.encoder = AutoModel.from_pretrained('facebook/wav2vec2-base', device_map="cuda")
            for param in self.encoder.parameters():
                param.requires_grad = False

        # Layer 1 - Linear Projection
        # Reduces voc2vec output from 768 → 384
        self.projection = nn.Linear(768, 384)

        # Layer 2 - 1D CNN
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=384, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Layer 3 - 2 layers of BiLSTM
        self.bilstm = nn.LSTM(
            input_size=128,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # Layer 4 - Attention Pooling
        # Focuses on most important parts of audio
        self.attention = AttentionPooling(hidden_size=512)

        # Layer 5 - Dense Layers
        self.dense = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Layer 6 - 5 Output Heads
        # One head per stutter type!
        self.head_prolongation = nn.Linear(128, 1)
        self.head_block        = nn.Linear(128, 1)
        self.head_soundrep     = nn.Linear(128, 1)
        self.head_wordrep      = nn.Linear(128, 1)
        self.head_interjection = nn.Linear(128, 1)

    def forward(self, x):
        # x shape → (batch, samples) raw audio waveform

        x = torch.squeeze(x)
        
        #  STEP 1 — Encoder
        with torch.no_grad():
            x = self.encoder(x).last_hidden_state

        # STEP 2 — Linear Projection
        x = self.projection(x)             # → (batch, time_steps, 384)
        x = torch.relu(x)

        # STEP 3 — 1D CNN
        x = x.permute(0, 2, 1)            # → (batch, 384, time_steps)
        x = self.cnn(x)                    # → (batch, 128, time_steps)
        x = x.permute(0, 2, 1)            # → (batch, time_steps, 128)

        # STEP 4 — BiLSTM
        x, _ = self.bilstm(x)             # → (batch, time_steps, 512)

        # STEP 5 — Attention Pooling
        x = self.attention(x)             # → (batch, 512)

        # STEP 6 — Dense Layers
        x = self.dense(x)                 # → (batch, 128)

        return {
            'prolongation': torch.sigmoid(self.head_prolongation(x).squeeze()),
            'block':        torch.sigmoid(self.head_block(x).squeeze()),
            'soundrep':     torch.sigmoid(self.head_soundrep(x)).squeeze(),
            'wordrep':      torch.sigmoid(self.head_wordrep(x)).squeeze(),
            'interjection': torch.sigmoid(self.head_interjection(x)).squeeze()
        }


# Initialize Model
encoder_name = 'wav2vec'
model  = StutterNet(encoder_type=encoder_name).to(device)
# print(model)

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 2199.24it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from torch.utils.tensorboard import SummaryWriter
# early_stopper = EarlyStopping(patience=5, delta=0.01)
writer = SummaryWriter(comment="encoder " + encoder_name)

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

eval_accu=[]
eval_losses=[]

classes = ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']

scaler = GradScaler(device=device)

def train(epoch):
        # print('\nEpoch : %d'%epoch)

        model.train()

        running_loss = {head_name: 0 for head_name in classes}
        correct = {head_name: 0 for head_name in classes}
        total = {head_name: 0 for head_name in classes}

        for data in tqdm(train_loader, desc="Training", total=len(train_loader)):
        # for data in train_loader:
            
            inputs = data[0].to(device)
            
            targets = {}
            for head_name in classes:
                targets[head_name] = data[1][head_name].to(device)

            # forward pass
            optimizer.zero_grad()
            outputs = model(inputs)
            
            loss = 0
            for head_name in classes:
                loss += criterion(outputs[head_name], targets[head_name])
                running_loss[head_name] += loss.item()
            # loss = loss / 5  # Average loss across heads
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # print(outputs)
            # Compute Training Accuracy
            for head_name in classes:
                predicted_labels = torch.round(outputs[head_name])
                # predicted_labels = outputs[head_name]
                total[head_name] += targets[head_name].size(0)
                correct[head_name] += predicted_labels.eq(targets[head_name]).sum().item()
        
        for head_name in classes:
            accu = 100.*correct[head_name] / total[head_name]
            train_loss = running_loss[head_name] / len(train_loader)
            # visualisation
            writer.add_scalar(head_name + " Loss/train", train_loss, epoch);
            writer.add_scalar(head_name + " Accuracy/train", accu, epoch);
            
            

def eval(epoch):
    model.eval()

    running_loss = {head_name: 0 for head_name in classes}
    correct = {head_name: 0 for head_name in classes}
    total = {head_name: 0 for head_name in classes}

    with torch.no_grad():
        # for data in tqdm(valid_loader, desc="Validating", total=len(valid_loader)):
        for data in valid_loader:
            inputs = data[0].to(device)
            
            targets = {}
            for head_name in classes:
                targets[head_name] = data[1][head_name].to(device)

            outputs = model(inputs)
            
            loss = 0
            outputs_matrix = torch.stack([
                outputs['prolongation'],
                outputs['block'],
                outputs['soundrep'],
                outputs['wordrep'],
                outputs['interjection']
            ], dim=1)

            targets_matrix = torch.stack([
                targets['prolongation'],
                targets['block'],
                targets['soundrep'],
                targets['wordrep'],
                targets['interjection']
            ], dim=1)

            # Single vectorized loss call
            loss = criterion(outputs_matrix, targets_matrix)

            for head_name in classes:
                predicted_labels = torch.round(outputs[head_name])
                # print(predicted_labels)
                total[head_name] += targets[head_name].size(0)
                correct[head_name] += predicted_labels.eq(targets[head_name]).sum().item()
    
    for head_name in classes:
        validate_loss = running_loss[head_name] / len(valid_loader)

        accu=100.*correct[head_name] / total[head_name]
        
        # visualisation
        writer.add_scalar(head_name + " Loss/eval", loss, epoch);
        writer.add_scalar(head_name + " Accuracy/eval", accu, epoch);  

        eval_accu.append(accu)
        eval_losses.append(validate_loss)

        # print(head_name + ' Loss:\t %.3f | Accuracy: %.3f'%(validate_loss, accu))
    return validate_loss

In [28]:
epochs=num_epochs
train_start_time = datetime.now()

for epoch in tqdm(range(1,epochs+1), desc="Epoch", total=epochs+1):
# for epoch in range(1, epochs + 1):
    train(epoch)
    eval_loss = eval(epoch)
    torch.cuda.empty_cache()
    # if early_stopper(eval_loss, model):
    #     print('Eval performance not improving; would be early stopping now!')
    #     model.load_state_dict(early_stopper.best_model)
    #     # TODO: figure out why break makes our number of predictions go to zero :|
    #     # break

train_end_time = datetime.now()
print("\nFinished training model in " + str(train_end_time - train_start_time) + " seconds.\n")

##################################################################################################
torch.save(model, 'DisfluencyDetector.pth')
##################################################################################################

Epoch:   0%|          | 0/151 [00:36<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
## TEST MODEL ##
# test_loader = DataLoader(dataset=test_dataset,
#                         batch_size=batch_size,
#                         shuffle=True,
#                         num_workers=0)

# model.load_state_dict(early_stopper.best_model)
model.eval()
output = []

with torch.no_grad():
    total = {head_name:0 for head_name in classes}
    correct = {head_name:0 for head_name in classes}
    
    # Compute F1 score, precision and recall
    n_correct = 0
    predicted_stutter = {head_name:0 for head_name in classes}
    labels_stutter = {head_name:0 for head_name in classes}
    correct_stutter = {head_name:0 for head_name in classes}

    running_loss = {head_name: 0 for head_name in classes}

    # final_label = []
    # final_predicted = []

    classToOutputMap = {
        'block': 'Block',
        'soundrep': 'SoundRep',
        'wordrep': 'WordRep',
        'interjection': 'Interjection',
        'prolongation': 'Prolongation'
    }

    exportColumns = {'Clip': []}
    
    for head_name in classes:
        exportColumns[classToOutputMap[head_name]] = []

    for data in test_loader:
        clip_names = data[2]
        inputs = data[0].to(device)

        exportColumns['Clip'] += clip_names

        targets = {}
        for head_name in classes:
                targets[head_name] = data[1][head_name].to(device)
                
        outputs = model(inputs)

        loss = 0
        for head_name in classes:
            loss += criterion(outputs[head_name], targets[head_name])
            running_loss[head_name] + loss.item()

            predicted_labels = torch.round(outputs[head_name])
            # print(predicted_labels)
            total[head_name] += targets[head_name].size(0)
            correct[head_name] += predicted_labels.eq(targets[head_name]).sum().item()

            for i in range (0, len(predicted_labels)) :
                if (predicted_labels[i] == 1) :
                    predicted_stutter[head_name] +=1
                if (predicted_labels[i] == 1) :
                    labels_stutter[head_name] +=1
                if ((predicted_labels[i] == 1) & (targets[head_name][i] == 1)):
                    correct_stutter[head_name] +=1
                if (predicted_labels[i] == targets[head_name][i]) :
                    n_correct = n_correct + 1

            print(exportColumns[classToOutputMap[head_name]])

            exportColumns[classToOutputMap[head_name]] += outputs[head_name].tolist()

    for key in exportColumns.keys():
        print(len(exportColumns[key]))

    exportDf = pd.DataFrame(exportColumns)
    exportDf.to_csv('results.csv')

    for head_name in classes: 
        for head_name in classes:
            accu = 100.*correct[head_name] / total[head_name]
            test_loss = running_loss[head_name] / len(train_loader)
            # visualisation
            writer.add_scalar(head_name + " Loss/test", loss, epoch);
            writer.add_scalar(head_name + " Accuracy/test", accu, epoch);
        
        # THESE METRICS ARE INCORRECT, REFER TO THE RUN LOGS INSTEAD
        print(head_name + '---------------------------------------------')
        predicted = predicted_stutter[head_name]
        print(f'Predicted : {predicted}%')
        print(f'Correct : {correct[head_name]}%')

        acc_test = 100*correct[head_name]/total[head_name]
        print(f'Accuracy: {acc_test} %')
        recall = correct[head_name]/ labels_stutter[head_name]
        print(f'Recall : {recall}')
        precision = correct[head_name] / predicted_stutter[head_name]
        print(f'Precision : {precision}')
        f1_score = 2 * precision * recall / (precision + recall)
        print(f'F1 Score : {f1_score}')

In [ ]:
exportColumns['Block']